# Napari

This script is used to create napari visualizations and animations.

This script can't be used on the UPGONPC2 when using Remote Desktop. Thus this script must either be ran from your personal computer or from the local UPGONPC2 directly.

In [28]:
import napari
import numpy as np
import tifffile
from napari_animation import Animation
from pathlib import Path
import sys
import tifffile
import skimage
import scipy
from napari_animation.easing import Easing


my_path = Path.cwd().parent.parent / "src"
sys.path.append(str(my_path))

from utils import utils

In [ ]:
ROOT = Path(r"Z:\data\current\TgCetnEos_CenSpark_H2B_4h-24h-48hpf\20260507CFHa_CetnEos_H2BmCherry_CS_48hpf\3e5")

VOL_PATH = ROOT/"20260507CFHa_CetnEos_H2BmCherry_CS_48hpf.lif - 3e5.tif"
SEGM_VOL_PATH = ROOT/"results"/"detection"/"raw"/"nucl_res"/"C2_detection_img.tif"

OUTPUT_PATH = ROOT/"results"/"animation"/"napari_animation.mp4"

In [25]:
vol_segmented = tifffile.imread(SEGM_VOL_PATH)

vol = vol_segmented[:,0]
labels = vol_segmented[:,1]

scale = utils.get_pixel_size(VOL_PATH)

Set up the images

In [26]:
viewer = napari.Viewer(ndisplay=3)  # ← 3D mode

image_layer = viewer.add_image(
    vol,
    name="nuclei",
    colormap="gray",
    depiction="plane",
    blending='translucent',
    #scale = scale
)
labels_layer = viewer.add_labels(
    labels,
    name="labels", blending='translucent',#scale = scale
)

This animation is based on this [example](https://napari.org/napari-animation/gallery/layer_planes.html)

In [24]:

animation = Animation(viewer)

viewer.camera.angles = (-18.23797054423494, 110.97404742075617, 141.96173085742896)
viewer.camera.zoom *= 2
speed = 120

denoised = scipy.ndimage.median_filter(vol, size=3)
th_nuclei = denoised > skimage.filters.threshold_li(denoised)
th_nuclei = skimage.morphology.remove_small_holes(th_nuclei, 20**3)
labels_data = skimage.measure.label(th_nuclei)

def replace_labels_data():
    z_cutoff = int(image_layer.plane.position[0])
    new_labels_data = labels_data.copy()
    new_labels_data[z_cutoff:] = 0
    labels_layer.data = new_labels_data


labels_layer.visible = False
image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=speed)

image_layer.plane.position = (59, 0, 0)
animation.capture_keyframe(steps=speed)

image_layer.plane.position = (0, 0, 0)

animation.capture_keyframe(steps=speed)

image_layer.plane.events.position.connect(replace_labels_data)
labels_layer.visible = True
labels_layer.experimental_clipping_planes = [{
    "position": (0, 0, 0),
    "normal": (-1, 0, 0),  # point up in z (i.e: show stuff above plane)
}]

# access first plane, since it's a list
labels_layer.experimental_clipping_planes[0].position = (59, 0, 0)
animation.capture_keyframe(steps=speed)

image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=speed)

animation.animate(OUTPUT_PATH, canvas_only=True)
image_layer.plane.position = (0, 0, 0)

Rendering frames...


100%|██████████| 481/481 [01:25<00:00,  5.64it/s]


In [29]:

animation = Animation(viewer)

# ── Setup ──────────────────────────────────────────────────────────────────────
Z_MAX   = labels_data.shape[0]   # total Z depth
speed   = 80                     # steps between keyframes (lower = faster)

# Good isometric angle to see the 3D structure clearly
CAMERA_3D      = (-25, 145, 135)
CAMERA_3D_SIDE = (-15, 200, 135)   # slightly rotated for the reveal
ZOOM           = viewer.camera.zoom * 1.8

viewer.camera.zoom   = ZOOM
viewer.camera.angles = CAMERA_3D

# ── Helper ─────────────────────────────────────────────────────────────────────
def set_clip(z):
    """Set the clipping plane position on both layers."""
    labels_layer.experimental_clipping_planes[0].position = (z, 0, 0)
    image_layer.plane.position = (z, 0, 0)

def update_labels_on_plane(event=None):
    z = int(image_layer.plane.position[0])
    new = labels_data.copy()
    new[z:] = 0
    labels_layer.data = new

# ── Initial state: raw volume only, plane at bottom ───────────────────────────
labels_layer.visible = False
image_layer.visible  = True
image_layer.plane.position = (0, 0, 0)

labels_layer.experimental_clipping_planes = [{
    "position": (0, 0, 0),
    "normal":   (-1, 0, 0),
}]

# Keyframe 0: static opening shot — raw data, plane at z=0
animation.capture_keyframe(steps=speed // 2)

# ── Act 1: sweep the plane through the raw volume ─────────────────────────────
# Show what the raw data looks like slice by slice
image_layer.plane.position = (Z_MAX, 0, 0)
animation.capture_keyframe(steps=speed * 2, ease=Easing.SINE)

# Hold at the bottom briefly
animation.capture_keyframe(steps=speed // 2)

# Sweep back to top
image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=speed * 2, ease=Easing.SINE)

# Hold at top
animation.capture_keyframe(steps=speed // 2)

# ── Act 2: reveal segmentation alongside raw ──────────────────────────────────
# Connect plane movement to label update
image_layer.plane.events.position.connect(update_labels_on_plane)

labels_layer.visible  = True
labels_layer.opacity  = 0.0   # fade in from transparent
animation.capture_keyframe(steps=speed // 2)

labels_layer.opacity  = 0.6   # fade in segmentation
animation.capture_keyframe(steps=speed, ease=Easing.CUBIC)

# Now sweep through with both raw and segmentation visible
image_layer.plane.position = (Z_MAX, 0, 0)
set_clip(Z_MAX)
animation.capture_keyframe(steps=speed * 2, ease=Easing.SINE)

# Hold at bottom
animation.capture_keyframe(steps=speed // 2)

# Sweep back
image_layer.plane.position = (0, 0, 0)
set_clip(0)
animation.capture_keyframe(steps=speed * 2, ease=Easing.SINE)

# ── Act 3: show full segmentation with a 360° rotation ────────────────────────
# Disconnect plane callback, show full labels at once
image_layer.plane.events.position.disconnect(update_labels_on_plane)

image_layer.visible  = False   # hide raw — show segmentation only
labels_layer.data    = labels_data   # restore full labels (no clipping)
labels_layer.experimental_clipping_planes = []  # remove clipping plane
labels_layer.opacity = 0.85
animation.capture_keyframe(steps=speed, ease=Easing.CUBIC)

# 360° rotation — 6 keyframes evenly spaced
n_rotation_steps = 6
for i in range(1, n_rotation_steps + 1):
    angle_y = 145 + (360 / n_rotation_steps) * i
    viewer.camera.angles = (-25, angle_y % 360, 135)
    animation.capture_keyframe(steps=speed, ease=Easing.SINE)

# ── Act 4: final shot — raw + segmentation together, no clipping ──────────────
image_layer.visible  = True
image_layer.opacity  = 0.35   # ghost the raw so segmentation is clear
labels_layer.opacity = 0.75
viewer.camera.angles = CAMERA_3D
animation.capture_keyframe(steps=speed, ease=Easing.CUBIC)

# Hold on final frame
animation.capture_keyframe(steps=speed // 2)

# ── Export ─────────────────────────────────────────────────────────────────────
animation.animate(OUTPUT_PATH, canvas_only=True, fps=30)

Rendering frames...


100%|██████████| 1561/1561 [01:11<00:00, 21.71it/s]


In [ ]:
viewer.camera.angles = (-18.23797054423494, 41.97404742075617, 141.96173085742896)
viewer.camera.zoom *= 2
speed = 120

def replace_labels_data():
    z_cutoff = int(image_layer.plane.position[0])
    new_labels_data = labels_data.copy()
    new_labels_data[z_cutoff:] = 0
    labels_layer.data = new_labels_data


labels_layer.visible = False
image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=speed)

image_layer.plane.position = (59, 0, 0)
animation.capture_keyframe(steps=speed)

image_layer.plane.position = (0, 0, 0)

animation.capture_keyframe(steps=speed)

image_layer.plane.events.position.connect(replace_labels_data)
labels_layer.visible = True
labels_layer.experimental_clipping_planes = [{
    "position": (0, 0, 0),
    "normal": (-1, 0, 0),  # point up in z (i.e: show stuff above plane)
}]

image_layer.plane.position = (59, 0, 0)
# access first plane, since it's a list
labels_layer.experimental_clipping_planes[0].position = (59, 0, 0)
animation.capture_keyframe(steps=speed)

image_layer.plane.position = (0, 0, 0)
animation.capture_keyframe(steps=speed)

animation.animate(OUTPUT_PATH, canvas_only=True)
image_layer.plane.position = (0, 0, 0)

Rendering frames...


100%|██████████| 721/721 [07:22<00:00,  1.63it/s]


In [ ]:
viewer.dims.ndisplay = 3
viewer.camera.angles = (0.0, 0.0, 90.0)
animation.capture_keyframe()
viewer.camera.zoom = 2.4
animation.capture_keyframe()
viewer.camera.angles = (-7.0, 15.7, 62.4)
animation.capture_keyframe(steps=60)
viewer.camera.angles = (2.0, -24.4, -36.7)
animation.capture_keyframe(steps=60)
viewer.reset_view()
viewer.camera.angles = (0.0, 0.0, 90.0)
animation.capture_keyframe()
animation.animate(OUTPUT_PATH, canvas_only=False)

Rendering frames...


 36%|███▋      | 137/376 [00:10<00:18, 12.71it/s]c:\Users\ovola\anaconda3\envs\napari-env\Lib\site-packages\napari_animation\interpolation\base_interpolation.py:169: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  return c_rotation.as_euler("ZYX", degrees=True)
c:\Users\ovola\anaconda3\envs\napari-env\Lib\site-packages\napari\_vispy\camera.py:58: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  angles = rotation.as_euler('yzx', degrees=True)
100%|██████████| 376/376 [00:26<00:00, 14.09it/s]
